# Data Preprocessing

This notebook prepares the cleaned dataset for machine learning by handling the remaining missing values, encoding categorical variables, scaling numerical features, splitting the dataset, and saving the processed data.

In [20]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

In [21]:
load_dotenv()

BASE_PATH = os.getenv("BASE_PATH")

df = pd.read_parquet(
    f"{BASE_PATH}/data/processed/clean_weather_training_data.parquet"
)

df.head()

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,Albury,22.900000,22.900000,0.6,NaN,NaN,W,44.0,W,WNW,...,71.0,22.0,1007.700012,1007.099976,8,7,16.900000,21.799999,No,False
1,Albury,25.100000,25.100000,0.0,NaN,NaN,WNW,44.0,NNW,WSW,...,44.0,25.0,1010.599976,1007.799988,8,7,17.200001,24.299999,No,False
2,Albury,32.299999,32.299999,1.0,NaN,NaN,W,41.0,ENE,NW,...,82.0,33.0,1010.799988,1006.000000,7,8,17.799999,29.700001,No,False
3,Albury,29.700001,29.700001,0.2,NaN,NaN,WNW,56.0,W,W,...,55.0,23.0,1009.200012,1005.400024,8,7,20.600000,28.900000,No,False
4,Albury,26.700001,26.700001,0.0,NaN,NaN,W,35.0,SSE,W,...,48.0,19.0,1013.400024,1010.099976,8,7,16.299999,25.500000,No,False


In [22]:
df.info()

print(df.shape)

df.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 99484 entries, 0 to 99483
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   Location       99484 non-null  category
 1   MinTemp        99484 non-null  float32 
 2   MaxTemp        99484 non-null  float32 
 3   Rainfall       99484 non-null  float32 
 4   Evaporation    56985 non-null  float32 
 5   Sunshine       52199 non-null  float32 
 6   WindGustDir    99484 non-null  category
 7   WindGustSpeed  99484 non-null  float32 
 8   WindDir9am     99484 non-null  category
 9   WindDir3pm     99484 non-null  category
 10  WindSpeed9am   99484 non-null  float32 
 11  WindSpeed3pm   99484 non-null  float32 
 12  Humidity9am    99484 non-null  float32 
 13  Humidity3pm    99484 non-null  float32 
 14  Pressure9am    99484 non-null  float32 
 15  Pressure3pm    99484 non-null  float32 
 16  Cloud9am       99484 non-null  Int8    
 17  Cloud3pm       99484 non-null  Int8    
 1

Location             0
MinTemp              0
MaxTemp              0
Rainfall             0
Evaporation      42499
Sunshine         47285
WindGustDir          0
WindGustSpeed        0
WindDir9am           0
WindDir3pm           0
WindSpeed9am         0
WindSpeed3pm         0
Humidity9am          0
Humidity3pm          0
Pressure9am          0
Pressure3pm          0
Cloud9am             0
Cloud3pm             0
Temp9am              0
Temp3pm              0
RainToday            0
RainTomorrow         0
dtype: int64

In [23]:
df = df.drop(columns=["Sunshine", "Evaporation"])

df.shape

(99484, 20)

In [24]:
categorical_cols = df.select_dtypes(include="category").columns

df[categorical_cols] = df[categorical_cols].astype(str)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99484 entries, 0 to 99483
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Location       99484 non-null  str    
 1   MinTemp        99484 non-null  float32
 2   MaxTemp        99484 non-null  float32
 3   Rainfall       99484 non-null  float32
 4   WindGustDir    99484 non-null  str    
 5   WindGustSpeed  99484 non-null  float32
 6   WindDir9am     99484 non-null  str    
 7   WindDir3pm     99484 non-null  str    
 8   WindSpeed9am   99484 non-null  float32
 9   WindSpeed3pm   99484 non-null  float32
 10  Humidity9am    99484 non-null  float32
 11  Humidity3pm    99484 non-null  float32
 12  Pressure9am    99484 non-null  float32
 13  Pressure3pm    99484 non-null  float32
 14  Cloud9am       99484 non-null  Int8   
 15  Cloud3pm       99484 non-null  Int8   
 16  Temp9am        99484 non-null  float32
 17  Temp3pm        99484 non-null  float32
 18  RainToday      99

## 5. Encode Categorical Variables

Machine learning algorithms require numerical inputs.

In this section, categorical features are transformed into numerical representations using:
- One-Hot Encoding for nominal categorical variables.
- Label Encoding for binary categorical variables.

In [25]:
categorical_cols = df.select_dtypes(include="object").columns

print(categorical_cols.tolist())

['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']


C:\Users\Farag\AppData\Local\Temp\ipykernel_8876\2801479450.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include="object").columns


In [26]:
from sklearn.preprocessing import OneHotEncoder

In [27]:
onehot_cols = [
    "Location",
    "WindGustDir",
    "WindDir9am",
    "WindDir3pm",
]

In [28]:
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

In [29]:
encoded = encoder.fit_transform(df[onehot_cols])

In [30]:
encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(onehot_cols),
    index=df.index
)

encoded_df.head()

,Location_Adelaide,Location_Albany,Location_Albury,Location_AliceSprings,Location_BadgerysCreek,Location_Ballarat,Location_Bendigo,Location_Brisbane,Location_Cairns,Location_Canberra,...,WindDir3pm_NNW,WindDir3pm_NW,WindDir3pm_S,WindDir3pm_SE,WindDir3pm_SSE,WindDir3pm_SSW,WindDir3pm_SW,WindDir3pm_W,WindDir3pm_WNW,WindDir3pm_WSW
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [31]:
df = pd.concat([df, encoded_df], axis=1)

df.head()

,Location,MinTemp,MaxTemp,Rainfall,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,WindSpeed9am,WindSpeed3pm,...,WindDir3pm_NNW,WindDir3pm_NW,WindDir3pm_S,WindDir3pm_SE,WindDir3pm_SSE,WindDir3pm_SSW,WindDir3pm_SW,WindDir3pm_W,WindDir3pm_WNW,WindDir3pm_WSW
0,Albury,22.900000,22.900000,0.6,W,44.0,W,WNW,20.0,24.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,Albury,25.100000,25.100000,0.0,WNW,44.0,NNW,WSW,4.0,22.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,Albury,32.299999,32.299999,1.0,W,41.0,ENE,NW,7.0,20.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Albury,29.700001,29.700001,0.2,WNW,56.0,W,W,19.0,24.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,Albury,26.700001,26.700001,0.0,W,35.0,SSE,W,6.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [32]:
df.drop(columns=onehot_cols, inplace=True)

df.head()

,MinTemp,MaxTemp,Rainfall,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,...,WindDir3pm_NNW,WindDir3pm_NW,WindDir3pm_S,WindDir3pm_SE,WindDir3pm_SSE,WindDir3pm_SSW,WindDir3pm_SW,WindDir3pm_W,WindDir3pm_WNW,WindDir3pm_WSW
0,22.900000,22.900000,0.6,44.0,20.0,24.0,71.0,22.0,1007.700012,1007.099976,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,25.100000,25.100000,0.0,44.0,4.0,22.0,44.0,25.0,1010.599976,1007.799988,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,32.299999,32.299999,1.0,41.0,7.0,20.0,82.0,33.0,1010.799988,1006.000000,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,29.700001,29.700001,0.2,56.0,19.0,24.0,55.0,23.0,1009.200012,1005.400024,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,26.700001,26.700001,0.0,35.0,6.0,17.0,48.0,19.0,1013.400024,1010.099976,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [33]:
df["RainToday"] = df["RainToday"].map({
    "No": 0,
    "Yes": 1
})

In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99484 entries, 0 to 99483
Columns: 113 entries, MinTemp to WindDir3pm_WSW
dtypes: Int8(2), bool(1), float32(12), float64(97), int64(1)
memory usage: 79.4 MB


## 6. Split Features and Target

Separate the predictor variables (features) from the target variable before training machine learning models.

In [35]:
X = df.drop(columns=["RainTomorrow"])
y = df["RainTomorrow"]

print(X.shape)
print(y.shape)

(99484, 112)
(99484,)


## 7. Train-Test Split

The dataset is divided into training and testing subsets.

- Training Set (80%) → Used to train the machine learning model.
- Testing Set (20%) → Used to evaluate model performance on unseen data.

In [36]:
from sklearn.model_selection import train_test_split

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [38]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (79587, 112)
X_test : (19897, 112)
y_train: (79587,)
y_test : (19897,)


## 8. Save Preprocessed Data

Save the preprocessed datasets for the next stage of the machine learning pipeline.

In [39]:
from pathlib import Path

processed_path = Path(BASE_PATH) / "data" / "processed"

X_train.to_parquet(processed_path / "X_train.parquet")
X_test.to_parquet(processed_path / "X_test.parquet")

y_train.to_frame().to_parquet(processed_path / "y_train.parquet")
y_test.to_frame().to_parquet(processed_path / "y_test.parquet")

print("Preprocessed datasets saved successfully.")

Preprocessed datasets saved successfully.
